#### Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Path Management
current_dir = os.getcwd()
file_name = 'shell_katy_delivery_2025_raw.csv'

if 'notebooks' in current_dir:
    raw_path = os.path.join('..', 'data', 'raw', file_name)
else:
    raw_path = os.path.join('data', 'raw', file_name)

# 2. Load the Dataset
if os.path.exists(raw_path):
    df = pd.read_csv(raw_path)
    print(f"Successfully loaded {len(df)} records for {df['phase'].iloc[0]}")
else:
    print(f"ERROR: File not found at {raw_path}.")

# 3. Industry Standards for Diesel
BASE_TEMP_C = 15.0       # Standard Reference Temperature
BETA = 0.00084           # Thermal Expansion Coefficient for Diesel

Successfully loaded 60 records for 1. Pre-Trip Inspection


#### The Core Physics Engine (The VCF Function)

In [2]:
def calculate_vcf(temp_actual):
    """
    Calculates the Volume Correction Factor (VCF) based on ASTM D1250.
    VCF = 1 / (1 + BETA * (Temp_Actual - Temp_Base))
    """
    return 1 / (1 + BETA * (temp_actual - BASE_TEMP_C))

# Apply the VCF to the entire dataframe
df['vcf'] = df['temp_c'].apply(calculate_vcf)

# Calculate NET VOLUME (The 'True' amount of fuel)
df['net_volume_l'] = df['gross_volume_l'] * df['vcf']

# Calculate the 'Correction Delta' (How much the volume 'lied' due to heat)
df['thermal_expansion_l'] = df['gross_volume_l'] - df['net_volume_l']

print("Thermal correction applied to all telemetry points.")
df[['phase', 'temp_c', 'gross_volume_l', 'net_volume_l', 'thermal_expansion_l']].head(10)

Thermal correction applied to all telemetry points.


,phase,temp_c,gross_volume_l,net_volume_l,thermal_expansion_l
0,1. Pre-Trip Inspection,18.98,0.00,0.000000,0.000000
1,1. Pre-Trip Inspection,19.03,0.00,0.000000,0.000000
2,1. Pre-Trip Inspection,19.19,0.00,0.000000,0.000000
3,2. Departure to Depot,19.32,0.00,0.000000,0.000000
4,2. Departure to Depot,19.27,0.00,0.000000,0.000000
5,2. Departure to Depot,19.48,0.00,0.000000,0.000000
6,2. Departure to Depot,19.42,0.00,0.000000,0.000000
7,3. Entry & Loading,19.72,36142.66,35999.927487,142.732513
8,3. Entry & Loading,19.75,36143.76,36000.119523,143.640477
9,3. Entry & Loading,19.68,36141.60,36000.076499,141.523501


#### The Multi-Point Audit (Depot vs. Station)

In [3]:
# 1. Extract the 'Load' volume (End of loading phase)
load_data = df[df['phase'] == "3. Entry & Loading"].iloc[-1]
v_loaded = load_data['net_volume_l']

# 2. Extract the 'Arrival' volume (Start of arrival phase at Katy)
arrival_data = df[df['phase'] == "6. Entry & Stabilization (Station)"].iloc[0]
v_arrived = arrival_data['net_volume_l']

# 3. Perform the Audit
net_variance = v_arrived - v_loaded
variance_pct = (net_variance / v_loaded) * 100

print(f"--- OFFICIAL AUDIT: SHELL DEER PARK TO KATY ---")
print(f"Net Volume Loaded:  {v_loaded:,.2f} L")
print(f"Net Volume Arrived: {v_arrived:,.2f} L")
print(f"-----------------------------------------------")
print(f"Total Net Variance: {net_variance:,.2f} L")
print(f"Variance Percentage: {variance_pct:.4f}%")

# Industry Rule: If variance > 0.3%, flag for theft/leak investigation
if abs(variance_pct) > 0.3:
    print("STATUS: INVESTIGATION REQUIRED (Threshold Exceeded)")
else:
    print("STATUS: OPERATIONAL COMPLIANCE PASSED")

--- OFFICIAL AUDIT: SHELL DEER PARK TO KATY ---
Net Volume Loaded:  35,999.99 L
Net Volume Arrived: 36,000.01 L
-----------------------------------------------
Total Net Variance: 0.02 L
Variance Percentage: 0.0000%
STATUS: OPERATIONAL COMPLIANCE PASSED


#### Exporting Audited Data for Visualization

In [5]:
# Save to the 'processed' directory
if 'notebooks' in os.getcwd():
    out_path = '../data/processed/shell_katy_audited_2025.csv'
else:
    out_path = 'data/processed/shell_katy_audited_2025.csv'

os.makedirs(os.path.dirname(out_path), exist_ok=True)
df.to_csv(out_path, index=False)

print(f"Processed data saved at: {out_path}")

Processed data saved at: ../data/processed/shell_katy_audited_2025.csv
